# Telephone Audio Generator Experiments

## Libraries

In [3]:
# Librerías
import subprocess
from pydub import AudioSegment
from scipy.io import wavfile
import numpy as np
import IPython.display as ipd

In [4]:
# Función para reproducir audios
def play_audio(audio_path):
    return ipd.Audio(audio_path)

In [5]:
# Celda para escuchar audios
input_path = "/mnt/media/fair/audio/replay_attacks/datasets/ASVSpoof2017/ASVspoof2017_V2_dev/ASVspoof2017_V2_dev/D_1000001.wav"
play_audio(input_path)

## Implementation from scratch

In [6]:
# Función para ley de compresión Ley-A
def alaw_compression(audio):
    A = 87.6
    mask = np.abs(audio) < (1 / A)
    compressed = np.zeros_like(audio)

    # Primera condición: |x| < 1/A
    compressed[mask] = np.sign(audio[mask]) * (A * np.abs(audio)[mask] / 1 + np.log(A))
    
    # Segunda condición: 1/A <= |x| <= 1
    # compressed = np.sign(audio) * (1 + np.log(1 + 87.6 * np.abs(audio)) / np.log(1 + 87.6)) / 2
    compressed[~mask] = np.sign(audio[~mask]) * (1 + np.log(A * np.abs(audio)[~mask]) / (1 + np.log(A)))
    
    return np.int16(compressed * 32767)

# Función para ley de compresión Ley-Mu
def mulaw_compression(audio):
    mu = 255
    compressed = np.sign(audio) * (np.log(1 + mu * np.abs(audio)) / np.log(1 + mu))
    return np.int16(compressed * 32767)

# Función para convertir a audio telefónico
def convert_to_telephone(
    input_path,
    output_path,
    law="alaw"
):
    # Cargar el audio
    audio = AudioSegment.from_wav(input_path)
    
    # Convertir a mono
    audio = audio.set_channels(1)
    
    # Reducir la frecuencia de muestreo a 8 kHz
    audio = audio.set_frame_rate(8000)

    # Filtrado para limitar la banda de frecuencias (300 Hz a 3400 Hz)
    audio = audio.low_pass_filter(3400).high_pass_filter(300)

    # Convertir a numpy array
    samples = np.array(audio.get_array_of_samples()) / 32768.0

    # Aplicar compresión logarítmica
    if law == "alaw":
        processed = alaw_compression(samples)
    elif law == "mulaw":
        processed = mulaw_compression(samples)
    else:
        raise ValueError("Ley no soportada")

    # Guardar el audio en formato WAV
    wavfile.write(
        filename=output_path,
        rate=8000,
        data=processed
    )
    print(f"Archivo convertido guardado en: {output_path}")

In [7]:
# Convertir y guardar el audio en formato telefónico (códec G.711 a 16 bits)
output_path_alaw = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_alaw.wav"
convert_to_telephone(
    input_path=input_path,
    output_path=output_path_alaw,
    law="alaw"
)

# Escuchar audio
play_audio(output_path_alaw)

Archivo convertido guardado en: /home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_alaw.wav


In [8]:
# Convertir y guardar el audio en formato telefónico (códec G.711 a 16 bits)
output_path_mulaw = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_mulaw.wav"
convert_to_telephone(
    input_path=input_path,
    output_path=output_path_mulaw,
    law="mulaw"
)

# Escuchar audio
play_audio(output_path_mulaw)

Archivo convertido guardado en: /home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_mulaw.wav


## Wav files implementation with ffmpeg

In [9]:
# Función para convertir a formato telefónico con ffmpeg
def convert_to_telephone_with_ffmpeg(
    input_path,
    output_path,
    codec='pcm_alaw'
):
    
    try:
        subprocess.run([
            'ffmpeg', '-y',
            '-i', input_path,
            '-ar', '8000',
            '-ac', '1',
            '-af', 'highpass=f=300, lowpass=f=3400',
            '-c:a', codec,
            output_path
        ], check=True)
        print(f'Archivo convertido guardado en: {output_path}')
    except subprocess.CalledProcessError:
        print('Error al convertir el audio con ffmpeg')


In [12]:
# Convertir y guardar el audio en formato telefónico (códec G.711 a 16 bits)
output_path_alaw_ffmpeg = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_alaw_ffmpeg.wav"
convert_to_telephone_with_ffmpeg(
    input_path=input_path,
    output_path=output_path_alaw_ffmpeg
)

# Escuchar audio
play_audio(output_path_alaw_ffmpeg)

Archivo convertido guardado en: /home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_alaw_ffmpeg.wav


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena

In [13]:
# Convertir y guardar el audio en formato telefónico (códec G.711 a 16 bits)
output_path_mulaw_ffmpeg = "/home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_mulaw_ffmpeg.wav"
convert_to_telephone_with_ffmpeg(
    input_path=input_path,
    output_path=output_path_mulaw_ffmpeg,
    codec='pcm_mulaw'
)

# Escuchar audio
play_audio(output_path_mulaw_ffmpeg)

Archivo convertido guardado en: /home/pepelacasa/MM-PR-01568-audio_replay_attacks/audios/D_1000001_telephone_mulaw_ffmpeg.wav


ffmpeg version 6.1.1-3ubuntu5 Copyright (c) 2000-2023 the FFmpeg developers
  built with gcc 13 (Ubuntu 13.2.0-23ubuntu3)
  configuration: --prefix=/usr --extra-version=3ubuntu5 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --disable-omx --enable-gnutls --enable-libaom --enable-libass --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libglslang --enable-libgme --enable-libgsm --enable-libharfbuzz --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enable-libwebp --enable-libx265 --enable-libxml2 --enable-libxvid --enable-libzimg --ena